In [1]:
from __future__ import annotations

import json
from collections import defaultdict
from copy import copy, deepcopy
from pathlib import Path

import pandas as pd
from tqdm import tqdm

/home/samoed/Desktop/dialogmteb/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from datasets import Dataset, DatasetDict, Features, List, Value


def process_ds(data: list) -> Dataset:
    all_keys = set()
    for row in data:
        all_keys.update(row.keys())

    # Convert into a dataset, filling missing with None
    normalized_data = [{k: row.get(k, None) for k in all_keys} for row in data]

    part_ds = Dataset.from_list(normalized_data)
    return part_ds


def process_ds_dict(data: dict[str, dict]) -> DatasetDict:
    all_keys = set()
    for ds in data.values():
        for row in ds:
            all_keys.update(row.keys())

    features = {col: Value("string") for col in all_keys}
    features["dialog"] = List({"content": Value("string"), "role": Value("string")})
    features = Features(features)

    # Convert into a dataset, filling missing with None
    for name, ds in data.items():
        ds = [{k: row.get(k, "none") for k in all_keys} for row in ds]
        data[name] = Dataset.from_list(ds, features=features)
    return DatasetDict(data)

# TM1

In [2]:
split_path = Path("/home/samoed/Desktop/Taskmaster/TM-1-2019/train-dev-test")
splits = {}
for file in split_path.glob("*.csv"):
    split = file.name.split(".")[0]

    df = pd.read_csv(file, index_col=None, header=None)
    splits[split] = df[0].values.tolist()

In [3]:
with open("/home/samoed/Desktop/Taskmaster/TM-1-2019/self-dialogs.json") as f:
    self_dialog = json.load(f)

with open("/home/samoed/Desktop/Taskmaster/TM-1-2019/woz-dialogs.json") as f:
    woz_dialog = json.load(f)

KeyboardInterrupt: 

In [ ]:
# instructions = {}

# for file in Path("/home/samoed/Desktop/Taskmaster/TM-1-2019/instructions").glob("*.txt"):
#     instruction_id = file.name.split('.')[0]
#     with file.open() as f:
#         instructions[instruction_id] = f.read()

### WOZ

In [ ]:
full_data = []
part_data = defaultdict(list)
for row in tqdm(woz_dialog):
    conversation_id = row["conversation_id"]
    instruction_id = row["instruction_id"]
    dialog = []
    for replic in row["utterances"]:
        dialog.append(
            {
                "role": replic["speaker"].lower(),
                "content": replic["text"],
            }
        )

        cur_data = {
            "dialog": dialog,
            "conversation_id": conversation_id,
            "instruction_id": instruction_id,
        }

        annotations = {}
        if "segments" in replic:
            for segment in replic["segments"]:
                val = deepcopy(segment["annotations"][0]["name"])
                annotations[val] = segment["text"]

        full_data.append({**cur_data, "annotation": annotations})
        if len(annotations) > 0:
            subset = list(annotations.keys())[0].split(".")[0]
            for col, v in annotations.items():
                cur_data[col] = v
            part_data[subset].append(deepcopy(cur_data))

100%|██████████| 5507/5507 [00:01<00:00, 3034.10it/s]


In [ ]:
full_ds = process_ds(full_data)
part_ds = process_ds_dict(part_data)

In [ ]:
part_data.keys()

dict_keys(['movie_ticket', 'auto_repair', 'restaurant_reservation', 'pizza_ordering', 'uber_lyft', 'coffee_ordering'])

In [ ]:
part_ds

DatasetDict({
    movie_ticket: Dataset({
        features: ['movie_ticket.name.theater', 'movie_ticket.time.duration.accept', 'restaurant_reservation.type.seating', 'movie_ticket.time.duration.reject', 'conversation_id', 'movie_ticket.ticket_booking.accept', 'movie_ticket.type.screening.accept', 'movie_ticket.time.start.accept', 'movie_ticket.location.theater.reject', 'movie_ticket.price.ticket.accept', 'dialog', 'movie_ticket.num.tickets.accept', 'movie_ticket.location.theater.accept', 'movie_ticket.time.duration', 'movie_ticket.name.theater.accept', 'movie_ticket.time.start', 'movie_ticket.time.start.reject', 'movie_ticket.type.screening.reject', 'movie_ticket.price.ticket', 'movie_ticket.location.theater', 'movie_ticket.name.movie', 'movie_ticket.name.theater.reject', 'movie_ticket.type.screening', 'instruction_id', 'movie_ticket.num.tickets', 'movie_ticket.name.movie.accept', 'movie_ticket.num.tickets.reject', 'movie_ticket.price.ticket.reject', 'movie_ticket.name.movie.reject'],


In [ ]:
# full_ds.push_to_hub("DeepPavlov/TaskMaster1", config_name="woz_full")
# instruction_ds.push_to_hub("DeepPavlov/TaskMaster1", config_name="instructions")

In [ ]:
for subset, ds in part_ds.items():
    ds.push_to_hub("DeepPavlov/TaskMaster1", config_name=subset)

Creating parquet from Arrow format: 100%|██████████| 9/9 [00:00<00:00, 172.36ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  67%|██████▋   |  558kB /  829kB,  558kB/s  



Processing Files (1 / 1)                : 100%|██████████|  829kB /  829kB,  461kB/s  


Processing Files (1 / 1)                : 100%|██████████|  829kB /  829kB,  377kB/s  
New Data Upload                         : 100%|██████████|  829kB /  829kB,  377kB/s  
                                        : 100%|██████████|  829kB /  829kB            
Creating parquet from Arrow format: 100%|██████████| 7/7 [00:00<00:00, 225.80ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  84%|████████▍ |  549kB /  652kB,  686kB/s  










Processing Files (1 / 1)                : 100%|██████████|  652kB /  652kB,  217kB/s  

Processing Files (1 / 1)                :

#### SELF

In [ ]:
full_data = defaultdict(list)
part_data = defaultdict(lambda: defaultdict(list))

for row in tqdm(self_dialog):
    conversation_id = row["conversation_id"]
    instruction_id = row["instruction_id"]
    split_name = None
    for split, ids in splits.items():
        if conversation_id in ids:
            split_name = copy(split)
            break
    dialog = []
    for replic in row["utterances"]:
        dialog.append(
            {
                "role": replic["speaker"].lower(),
                "content": replic["text"],
            }
        )

        cur_data = {
            "dialog": dialog,
            "conversation_id": conversation_id,
            "instruction_id": instruction_id,
        }

        annotations = {}
        if "segments" in replic:
            for segment in replic["segments"]:
                val = deepcopy(segment["annotations"][0]["name"])
                annotations[val] = segment["text"]

        full_data[split].append({**cur_data, "annotation": annotations})
        if len(annotations) > 0:
            subset = list(annotations.keys())[0].split(".")[0]
            for col, v in annotations.items():
                cur_data[col] = v
            part_data[subset][split].append(deepcopy(cur_data))

100%|██████████| 7708/7708 [00:02<00:00, 2946.16it/s]


In [ ]:
for subset, data in part_data.items():
    ds = process_ds_dict(data)
    ds.push_to_hub("DeepPavlov/TaskMaster1", config_name=subset)

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 554.73ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████|  131kB /  131kB, 72.8kB/s  

Processing Files (1 / 1)                : 100%|██████████|  131kB /  131kB, 65.5kB/s  
New Data Upload                         : 100%|██████████|  131kB /  131kB, 65.5kB/s  
                                        : 100%|██████████|  131kB /  131kB            
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 466.84ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████|  137kB /  137kB, 68.5kB/s  

Processing Files (1 / 1)                : 100%|██████████|  137kB /  137kB, 62.2kB/s  
New Data Upload                         : 100%|██████████|  137kB /  137kB, 62.2kB/s  
                                        : 100%|████████

In [ ]:
ds["dev"].features

{'restaurant_reservation.type.seating': Value('string'),
 'restaurant_reservation.name.reservation': Value('string'),
 'conversation_id': Value('string'),
 'restaurant_reservation.name.reservation.accept': Value('string'),
 'restaurant_reservation.num.guests.reject': Value('string'),
 'restaurant_reservation.name.restaurant': Value('string'),
 'restaurant_reservation.num.guests.accept': Value('string'),
 'restaurant_reservation.type.seating.accept': Value('string'),
 'restaurant_reservation.time.reservation.accept': Value('string'),
 'dialog': List({'content': Value('string'), 'role': Value('string')}),
 'restaurant_reservation.reservation.reject': Value('string'),
 'restaurant_reservation.restaurant_reservation': Value('string'),
 'restaurant_reservation.location.restaurant.reject': Value('string'),
 'restaurant_reservation.name.reservation.reject': Value('string'),
 'restaurant_reservation.reservation.accept': Value('string'),
 'restaurant_reservation.location.restaurant.accept': Val

# TM2

In [7]:
list(Path("/home/samoed/Desktop/Taskmaster/TM-2-2020/data").glob("*.json"))

[PosixPath('/home/samoed/Desktop/Taskmaster/TM-2-2020/data/flights.json'),
 PosixPath('/home/samoed/Desktop/Taskmaster/TM-2-2020/data/food-ordering.json'),
 PosixPath('/home/samoed/Desktop/Taskmaster/TM-2-2020/data/hotels.json'),
 PosixPath('/home/samoed/Desktop/Taskmaster/TM-2-2020/data/movies.json'),
 PosixPath('/home/samoed/Desktop/Taskmaster/TM-2-2020/data/music.json'),
 PosixPath('/home/samoed/Desktop/Taskmaster/TM-2-2020/data/restaurant-search.json'),
 PosixPath('/home/samoed/Desktop/Taskmaster/TM-2-2020/data/sports.json')]

In [11]:
for file in Path("/home/samoed/Desktop/Taskmaster/TM-2-2020/data").glob("*.json"):
    with file.open() as f:
        data = json.load(f)

    subset = file.name.split(".")[0]

    full_data = []
    part_data = []

    for row in tqdm(data):
        conversation_id = row["conversation_id"]
        instruction_id = row["instruction_id"]
        dialog = []
        for replic in row["utterances"]:
            dialog.append(
                {
                    "role": replic["speaker"].lower(),
                    "content": replic["text"],
                }
            )

            cur_data = {
                "dialog": dialog,
                "conversation_id": conversation_id,
                "instruction_id": instruction_id,
            }

            annotations = {}
            if "segments" in replic:
                for segment in replic["segments"]:
                    val = deepcopy(segment["annotations"][0]["name"])
                    annotations[val] = segment["text"]

            full_data.append({**cur_data, "annotation": annotations})
            if len(annotations) > 0:
                for col, v in annotations.items():
                    cur_data[col] = v
                part_data.append(deepcopy(cur_data))

    ds = process_ds(data)
    ds.push_to_hub("DeepPavlov/TaskMaster2", config_name=subset)

Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 79.41ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  51%|█████     | 1.04MB / 2.03MB,  324kB/s  
Processing Files (0 / 1)                :  78%|███████▊  | 1.58MB / 2.03MB,  466kB/s  











Processing Files (1 / 1)                : 100%|██████████| 2.03MB / 2.03MB,  349kB/s  


Processing Files (1 / 1)                : 100%|██████████| 2.03MB / 2.03MB,  327kB/s  
New Data Upload                         : 100%|██████████|  990kB /  990kB,  160kB/s  
                                        : 100%|██████████| 2.03MB / 2.03MB            
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 159.45ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  29%|██▉       |  131kB /  452kB,  218kB/s  





Processing Files (1 / 1)                

# TM3